# 06 – EDA Mini-Project

You've learned:
- What EDA is and why it matters
- Matplotlib — line, bar, scatter, pie, subplots
- Histograms, KDE, boxplots, violin plots
- Scatter plots, correlation, heatmaps, pair plots
- How to write clear, specific insights

Now you put all of it together on a **new dataset** you haven't seen before.

This is what real EDA work looks like: a dataset lands in your hands, and you need to systematically understand it — its shape, its quality, its patterns, and the questions it raises.

Work through each section below. Each section asks you to produce charts **and** write the insight. The solution at the bottom of each section is there to compare — but only check it after you've tried.

---

## The Dataset — Retail Sales Records

You work for a retail analytics team. HR has given you a dataset of sales transactions from 2023 across five regions. The business wants to understand:

1. How are sales distributed across regions and salespersons?
2. Is there a relationship between experience and sales performance?
3. Which products are driving revenue?
4. Are there any data quality issues to flag?

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

rng = np.random.default_rng(99)

n = 300

salespersons = ["Rahul","Priya","Sneha","Karan","Meera","Vijay","Pooja","Amit","Neha","Ravi"]

df = pd.DataFrame({
    "Transaction_ID": range(5001, 5001 + n),
    "Date":       pd.date_range("2023-01-01", periods=n, freq="D")[:n],
    "Salesperson":rng.choice(salespersons, size=n),
    "Region":     rng.choice(["North","South","East","West","Central"], size=n,
                              p=[0.25,0.20,0.20,0.20,0.15]),
    "Product":    rng.choice(["Laptop","Phone","Tablet","Accessories","Monitor"],
                              size=n, p=[0.25,0.30,0.20,0.15,0.10]),
    "Units":      rng.integers(1, 11, size=n),
    "Unit_Price": rng.choice([55000,18000,25000,3000,32000], size=n),
    "Experience_Years": np.clip(rng.normal(5, 3, n).round(1), 0.5, 15),
    "Customer_Satisfaction": np.clip(rng.normal(3.8, 0.7, n).round(1), 1.0, 5.0),
})

# Compute revenue
df["Revenue"] = df["Units"] * df["Unit_Price"]

# Inject missing values
for col, frac in [("Customer_Satisfaction",0.06),("Experience_Years",0.04)]:
    idx = rng.choice(n, size=int(n*frac), replace=False)
    df.loc[idx, col] = np.nan

# Inject a few outlier revenue records
df.loc[rng.choice(n, 3, replace=False), "Revenue"] = rng.choice([800000, 750000, 900000], 3)

print(df.head())
print()
print("Shape:", df.shape)
print()
print(df.dtypes)

Take a few minutes to look at the raw data above before starting. What types of columns are there? What are the value ranges? Does anything immediately look off?

---
## Part 1 — Data Quality Check

**Your task:**
1. Count missing values per column (show count and percentage)
2. Check the Revenue column for outliers using the IQR method
3. Check whether any `Unit_Price` values look unusual (print all distinct unit prices)
4. Write a 3-sentence data quality summary

In [ ]:
# Your code here


<details>
<summary>Click to see solution</summary>

```python
# 1. Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print("Missing Values:")
print(pd.DataFrame({"Count": missing, "%": missing_pct})[missing > 0])
print()

# 2. Outlier check on Revenue
q1, q3 = df["Revenue"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
outliers = df[df["Revenue"] > upper_fence]
print(f"Revenue outliers (above ₹{upper_fence:,.0f}):")
print(outliers[["Transaction_ID","Salesperson","Product","Units","Unit_Price","Revenue"]])
print()

# 3. Distinct unit prices
print("Distinct Unit Prices:", sorted(df["Unit_Price"].unique()))
print()

# 4. Summary printed
print("DATA QUALITY SUMMARY")
print("-" * 50)
print(f"Two columns have missing values: Customer_Satisfaction ({missing_pct['Customer_Satisfaction']}%)")
print(f"and Experience_Years ({missing_pct['Experience_Years']}%). Both are minor and safe to impute.")
print(f"Three transactions have revenue above ₹{upper_fence:,.0f} — the IQR upper fence.")
print("These appear to be bulk orders and should be verified before modelling.")
```
</details>

---
## Part 2 — Distribution Analysis

**Your task:**
1. Plot the distribution of `Revenue` using `sns.histplot(kde=True)`. Add a mean and median line.
2. Plot the distribution of `Customer_Satisfaction`. Is it skewed?
3. Create boxplots of `Revenue` by `Region` — which region has the widest spread?
4. Write one insight for each chart (3 total)

In [ ]:
# Your code here


<details>
<summary>Click to see solution</summary>

```python
rev = df["Revenue"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1 — Revenue distribution
sns.histplot(rev, bins=25, kde=True, color="steelblue", edgecolor="white", alpha=0.7, ax=axes[0])
axes[0].axvline(rev.mean(),   color="firebrick", linestyle="--", lw=2, label=f"Mean: ₹{rev.mean():,.0f}")
axes[0].axvline(rev.median(), color="darkorange", linestyle="-", lw=2, label=f"Median: ₹{rev.median():,.0f}")
axes[0].set_title("Revenue Distribution", fontweight="bold")
axes[0].set_xlabel("Revenue (₹)")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# Chart 2 — Customer Satisfaction
sns.histplot(df["Customer_Satisfaction"].dropna(), bins=15, kde=True, 
             color="seagreen", edgecolor="white", alpha=0.7, ax=axes[1])
axes[1].set_title("Customer Satisfaction Distribution", fontweight="bold")
axes[1].set_xlabel("Satisfaction Score (1–5)")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

# Chart 3 — Revenue by Region (boxplot)
sns.boxplot(data=df, x="Region", y="Revenue", hue="Region", 
            palette="Set2", legend=False, ax=axes[2])
axes[2].set_title("Revenue by Region", fontweight="bold")
axes[2].set_xlabel("Region")
axes[2].set_ylabel("Revenue (₹)")
axes[2].grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle("Distribution Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/05_eda/project_distributions.png", dpi=120)
plt.close()
print("Charts saved.")
```
</details>

---
## Part 3 — Sales by Category

**Your task:**
1. Create a horizontal bar chart of **total revenue by Region** — sorted from highest to lowest
2. Create a grouped bar chart showing **total units sold by Product and Region** (pick 3 regions only to keep it readable)
3. Create a pie chart of revenue share by Product category
4. Which product and region combination is the most valuable? (use `groupby`)
5. Write the key finding in 2 sentences

In [ ]:
# Your code here


<details>
<summary>Click to see solution</summary>

```python
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Revenue by Region (horizontal bar)
region_rev = df.groupby("Region")["Revenue"].sum().sort_values()
axes[0].barh(region_rev.index, region_rev.values, color="steelblue", edgecolor="white")
for i, v in enumerate(region_rev.values):
    axes[0].text(v + 10000, i, f"₹{v/1e6:.2f}M", va="center", fontsize=9)
axes[0].set_title("Total Revenue by Region", fontweight="bold")
axes[0].set_xlabel("Revenue (₹)")
axes[0].grid(axis="x", linestyle="--", alpha=0.4)

# 2. Product revenue pie
prod_rev = df.groupby("Product")["Revenue"].sum()
axes[1].pie(prod_rev, labels=prod_rev.index, autopct="%1.1f%%",
            startangle=90, wedgeprops={"edgecolor":"white","linewidth":2},
            colors=sns.color_palette("Set2", len(prod_rev)))
axes[1].set_title("Revenue Share by Product", fontweight="bold")

# 3. Top product-region combo
combo = df.groupby(["Region","Product"])["Revenue"].sum().reset_index()
combo = combo.sort_values("Revenue", ascending=False)
print("Top 5 Product-Region combinations:")
print(combo.head())
print()

# Bar chart of top 8 combos
top8 = combo.head(8)
axes[2].barh(top8["Region"] + " — " + top8["Product"], 
             top8["Revenue"], color="darkorange", edgecolor="white")
axes[2].set_title("Top Product-Region Combos", fontweight="bold")
axes[2].set_xlabel("Revenue (₹)")
axes[2].grid(axis="x", linestyle="--", alpha=0.4)

fig.suptitle("Sales by Category", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/05_eda/project_categories.png", dpi=120)
plt.close()
print("Charts saved.")
```
</details>

---
## Part 4 — Relationships and Correlation

**Your task:**
1. Create a scatter plot of `Experience_Years` vs `Revenue`. Add a trend line. What's the correlation coefficient?
2. Create a scatter plot of `Customer_Satisfaction` vs `Revenue`. Colour by `Region`.
3. Create a correlation heatmap for `Units`, `Unit_Price`, `Revenue`, `Experience_Years`, `Customer_Satisfaction`.
4. Write 2 insights: one about the experience-revenue relationship, one about what the heatmap reveals.

In [ ]:
# Your code here


<details>
<summary>Click to see solution</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Experience vs Revenue
clean = df[["Experience_Years","Revenue"]].dropna()
r_exp = clean["Experience_Years"].corr(clean["Revenue"])
sns.regplot(data=clean, x="Experience_Years", y="Revenue",
            scatter_kws={"alpha":0.5,"color":"steelblue","edgecolors":"none","s":50},
            line_kws={"color":"firebrick","linewidth":2},
            ax=axes[0])
axes[0].set_title(f"Experience vs Revenue  (r = {r_exp:.2f})", fontweight="bold")
axes[0].set_xlabel("Experience (Years)")
axes[0].set_ylabel("Revenue (₹)")
axes[0].grid(linestyle="--", alpha=0.4)

# 2. Satisfaction vs Revenue by Region
palette = dict(zip(df["Region"].unique(), sns.color_palette("Set2",5)))
for region in df["Region"].unique():
    sub = df[df["Region"]==region][["Customer_Satisfaction","Revenue"]].dropna()
    axes[1].scatter(sub["Customer_Satisfaction"], sub["Revenue"],
                    color=palette[region], alpha=0.5, s=50, label=region, edgecolors="none")
axes[1].set_title("Satisfaction vs Revenue by Region", fontweight="bold")
axes[1].set_xlabel("Customer Satisfaction")
axes[1].set_ylabel("Revenue (₹)")
axes[1].legend(title="Region", fontsize=8)
axes[1].grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/project_scatter.png", dpi=120)
plt.close()

# Heatmap
num_cols = ["Units","Unit_Price","Revenue","Experience_Years","Customer_Satisfaction"]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, 
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap", fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/project_heatmap.png", dpi=120)
plt.close()
print(f"Experience vs Revenue: r = {r_exp:.3f}")
print("Heatmap saved.")
```
</details>

---
## Part 5 — Full EDA Summary

**Your task:** Write a complete EDA summary. Use the template from notebook 05_05 as a guide. Include:

1. **Data Overview** — rows, columns, missing values, data quality notes
2. **Distribution Insights** — revenue shape, customer satisfaction, any outliers
3. **Group Comparisons** — which region, product, salesperson stands out
4. **Relationships** — the most important correlations you found
5. **Questions for Further Investigation** — what the EDA raised but couldn't answer

In [ ]:
# Generate summary statistics to support your write-up
print("=== NUMBERS FOR YOUR SUMMARY ===")
print()
print(f"Dataset: {len(df)} transactions | {len(df.columns)} columns | 2023")
print()

# Missing
missing = df.isnull().sum()
print("Missing values:")
for col, cnt in missing[missing > 0].items():
    print(f"  {col}: {cnt} ({cnt/len(df)*100:.1f}%)")
print()

# Revenue
rev = df["Revenue"]
print(f"Revenue: mean=₹{rev.mean():,.0f}, median=₹{rev.median():,.0f}, max=₹{rev.max():,.0f}")
q1, q3 = rev.quantile([0.25, 0.75])
n_outliers = (rev > q3 + 1.5*(q3-q1)).sum()
print(f"Outliers above IQR fence: {n_outliers}")
print()

# By region
print("Revenue by Region:")
print(df.groupby("Region")["Revenue"].sum().sort_values(ascending=False).apply(lambda x: f"₹{x/1e6:.2f}M"))
print()

# By product
print("Revenue by Product:")
print(df.groupby("Product")["Revenue"].sum().sort_values(ascending=False).apply(lambda x: f"₹{x/1e6:.2f}M"))
print()

# Top correlations
num_cols = ["Units","Unit_Price","Revenue","Experience_Years","Customer_Satisfaction"]
corr = df[num_cols].corr()
rev_corr = corr["Revenue"].drop("Revenue").sort_values(key=abs, ascending=False)
print("Correlations with Revenue:")
for col, r in rev_corr.items():
    print(f"  {col:25}: r = {r:+.3f}")

**Write your summary here** (in a new cell or as markdown):

```
EDA SUMMARY — Retail Sales Dataset 2023
=========================================

DATA OVERVIEW
 - ...

DISTRIBUTION INSIGHTS
 - ...

GROUP COMPARISONS
 - ...

RELATIONSHIPS
 - ...

QUESTIONS FOR FURTHER INVESTIGATION
 - ...
```

In [ ]:
# Reference solution — full summary
print("""
EDA SUMMARY — Retail Sales Dataset 2023
=========================================

DATA OVERVIEW
 - 300 transactions across 2023, covering 5 regions, 5 products, and 10 salespersons
 - Two columns have minor missing values: Customer_Satisfaction (~6%) and Experience_Years (~4%)
   Both can be safely imputed with median values before analysis
 - 3 revenue outliers detected above the IQR fence — likely bulk orders; require business verification
   before inclusion in averages or model training

DISTRIBUTION INSIGHTS
 - Revenue is strongly right-skewed: the mean (₹{mean_rev:,.0f}) is significantly higher than
   the median (₹{med_rev:,.0f}) due to the outlier bulk orders
 - Customer Satisfaction is approximately normally distributed, centred around 3.8 out of 5,
   with most ratings between 3 and 5 — suggesting generally positive but not exceptional feedback
 - No unit price anomalies detected; all five price points correspond to distinct product categories

GROUP COMPARISONS
 - North is the highest-revenue region; Central is the lowest — consistent with its smallest
   share of transactions (15% of records)
 - Laptops and Phones together account for the majority of revenue despite Accessories having
   the lowest unit price, confirming that high-ticket items drive total revenue
 - Revenue per transaction varies widely within each region, suggesting salesperson performance
   is more influential than regional assignment

RELATIONSHIPS
 - Unit_Price is the strongest predictor of revenue (r ≈ 0.9+), as expected — higher-priced
   items drive higher transaction values
 - Experience_Years shows a weak positive correlation with Revenue ({r_exp_note}), suggesting
   that experienced salespeople are not dramatically outperforming newer ones on a per-transaction basis
 - Customer Satisfaction has near-zero correlation with Revenue — high-value transactions do
   not necessarily generate higher satisfaction scores

QUESTIONS FOR FURTHER INVESTIGATION
 - Do the 3 outlier bulk orders represent real sales or data errors? What product were they?
 - Is the weak experience-revenue link due to compensation structure (newer staff selling cheaper products)?
 - With quarterly data: is there a seasonal pattern in revenue across 2023?
 - Why is Customer Satisfaction uncorrelated with Revenue? Is it driven by product type instead?
""".format(
    mean_rev=rev.mean(),
    med_rev=rev.median(),
    r_exp_note="weak positive" if df[["Experience_Years","Revenue"]].dropna()["Experience_Years"].corr(
        df[["Experience_Years","Revenue"]].dropna()["Revenue"]) > 0 else "weak negative"
))

---
## What You've Built

In this mini-project you:
1. Loaded a new dataset cold and ran a systematic quality check
2. Plotted distributions and identified the right shape description for each
3. Compared groups using bar charts and boxplots
4. Found relationships with scatter plots and a heatmap
5. Wrote a structured summary with specific numbers and open questions

This is the full EDA workflow. Every project starts here — before modelling, before dashboards, before conclusions.

---

## What's Next?

| Section | Topic |
|---|---|
| 05_01 | What is EDA? |
| 05_02 | Matplotlib Basics |
| 05_03 | Histograms and Distributions |
| 05_04 | Scatter Plots and Correlation |
| 05_05 | Writing Insights |
| 05_06 | EDA Mini-Project (this notebook) |

**Up next: Section 06 — Statistics.**

Probability, distributions, hypothesis testing — the mathematical foundation behind what we've been doing visually.